In [ ]:
# 1. Login (Do this once)
from huggingface_hub import login
from huggingface_hub import create_repo, upload_folder
login("#hf_secret")

In [2]:
MODEL_ID = "Qwen/Qwen3-4B"
OUTPUT_DIR_25 = "/kaggle/working/wanda/Qwen3-4B-WANDA-25"
REPO_ID_25 = f"EdgeCompress01/Qwen3-4B-WANDA-25"

OUTPUT_DIR_50 = "/kaggle/working/wanda/Qwen3-4B-WANDA-50"
REPO_ID_50 = f"EdgeCompress01/Qwen3-4B-WANDA-50"

OUTPUT_DIR_70 = "/kaggle/working/wanda/Qwen3-4B-WANDA-70"
REPO_ID_70 = f"EdgeCompress01/Qwen3-4B-WANDA-70"
SEED = 42

# LLAMA

In [3]:
!pip install -U "transformers>=5.1.0" "accelerate>=1.1.0" "datasets>=3.0.0"
!git clone https://github.com/locuslab/wanda.git
%cd wanda

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 16.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.5/527.5 kB 14.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 23.1 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.12.0
    Uninstalling accelerate-1.12.0:
      Successfully uninstalled accelerate-1.12.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.2.0
    Uninstalling transformers-5.2.0:
      Successfully uninstalled transfor

In [4]:
import os
import re

file_path = '/kaggle/working/wanda/lib/data.py'

if os.path.exists(file_path):
    with open(file_path, 'r') as f:
        content = f.read()

    # Define the new, clean get_c4 function logic
    # This uses one shard for everything to avoid config and split errors
    new_get_c4_logic = """
def get_c4(nsamples, seed, seqlen, tokenizer):
    from datasets import load_dataset
    print("Loading C4 training shard...")
    traindata = load_dataset(
        'allenai/c4', 'en', 
        data_files={'train': 'en/c4-train.00000-of-01024.json.gz'}, 
        split='train', 
        verification_mode='no_checks'
    )
    valdata = traindata # Reuse train for val to avoid extra downloads/errors

    import random
    random.seed(seed)
    trainloader = []
    for _ in range(nsamples):
        while True:
            i = random.randint(0, len(traindata) - 1)
            trainenc = tokenizer(traindata[i]['text'], return_tensors='pt')
            if trainenc.input_ids.shape[1] > seqlen:
                break
        i = random.randint(0, trainenc.input_ids.shape[1] - seqlen - 1)
        j = i + seqlen
        inp = trainenc.input_ids[:, i:j]
        tar = inp.clone()
        tar[:, :-1] = -100
        trainloader.append((inp, tar))

    valenc = tokenizer(' '.join(traindata[:1100]['text']), return_tensors='pt')
    valenc = valenc.input_ids[:, :(256 * seqlen)]

    return trainloader, valenc
"""

    # Replace the entire get_c4 function
    # We look for the start of the function and replace until the return
    pattern = r"def get_c4\(.*?\n    return trainloader, valenc"
    fixed_content = re.sub(pattern, new_get_c4_logic, content, flags=re.DOTALL)

    with open(file_path, 'w') as f:
        f.write(fixed_content)
    print("✅ lib/data.py fully rebuilt. Training and Validation splits are now synchronized and fixed.")
else:
    print(f"❌ Error: File not found at {file_path}")

✅ lib/data.py fully rebuilt. Training and Validation splits are now synchronized and fixed.


In [ ]:
# # Install/Update the required quantization libraries
# !pip install -U bitsandbytes accelerate

## Change the Main file to have the wanda only

In [5]:
%%writefile main.py
import argparse
import os 
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from lib.prune import prune_wanda, check_sparsity

def get_llm(model_id, cache_dir):
    # Standard float16, but load to CPU first to save GPU memory
    model = AutoModelForCausalLM.from_pretrained(
        model_id, 
        dtype=torch.float16,
        cache_dir=cache_dir, 
        low_cpu_mem_usage=True, 
        device_map="cpu" 
    )
    model.seqlen = 2048 
    return model

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--model', type=str, required=True)
    parser.add_argument('--seed', type=int, default=0)
    parser.add_argument('--nsamples', type=int, default=128)
    parser.add_argument('--sparsity_ratio', type=float, default=0.25)
    parser.add_argument("--cache_dir", default="llm_weights", type=str )
    parser.add_argument('--save_model', type=str, default=None)
    
    args = parser.parse_args()
    np.random.seed(args.seed)
    torch.random.manual_seed(args.seed)

    model = get_llm(args.model, args.cache_dir)
    tokenizer = AutoTokenizer.from_pretrained(args.model, use_fast=False)
    device = torch.device("cuda:0")
    
    print("Pruning starts (Float16 mode)...")
    prune_wanda(args, model, tokenizer, device)

    print("*"*30)
    s_ratio = check_sparsity(model)
    print(f"Sparsity sanity check: {s_ratio*100:.2f}%")
    print("*"*30)

    if args.save_model:
        model.save_pretrained(args.save_model)
        tokenizer.save_pretrained(args.save_model)

if __name__ == '__main__':
    main()

Overwriting main.py


## Change the prune file

In [6]:
%%writefile lib/prune.py
import torch
import torch.nn as nn
import gc

def find_layers(module, layers=[nn.Linear], name=''):
    if type(module) in layers:
        return {name: module}
    res = {}
    for name1, child in module.named_children():
        res.update(find_layers(child, layers=layers, name=name + '.' + name1 if name != '' else name1))
    return res

def prepare_calibration_input(model, dataloader, device):
    layers = model.model.layers
    
    # Move necessary bookend layers to GPU for the initial forward pass
    model.model.embed_tokens.to(device)
    model.model.rotary_emb.to(device)
    
    dtype = next(iter(model.parameters())).dtype
    inps = torch.zeros((len(dataloader), model.seqlen, model.config.hidden_size), dtype=dtype, device='cpu')
    cache = {'i': 0, 'attention_mask': None, 'position_ids': None}

    class Catcher(nn.Module):
        def __init__(self, module):
            super().__init__()
            self.module = module
        def forward(self, inp, **kwargs):
            inps[cache['i']] = inp.detach().cpu()
            cache['i'] += 1
            if kwargs.get('attention_mask') is not None:
                cache['attention_mask'] = kwargs['attention_mask'].detach().cpu()
            if kwargs.get('position_ids') is not None:
                cache['position_ids'] = kwargs['position_ids'].detach().cpu()
            raise ValueError
            
    layers[0] = Catcher(layers[0].to(device))
    for batch in dataloader:
        try:
            with torch.no_grad():
                # Input must match the embedding layer's device
                model(batch[0].to(device))
        except ValueError:
            pass
            
    # Clean up after catching inputs
    layers[0] = layers[0].module.cpu()
    model.model.embed_tokens.cpu()
    torch.cuda.empty_cache()
    
    outs = torch.zeros_like(inps)
    return inps, outs, cache['attention_mask'], cache['position_ids']

def prune_wanda(args, model, tokenizer, device, prune_n=0, prune_m=0):
    from .data import get_loaders
    print("loading calibration data")
    dataloader, _ = get_loaders("c4", nsamples=args.nsamples, seed=args.seed, seqlen=model.seqlen, tokenizer=tokenizer)
    
    inps, outs, attention_mask, position_ids = prepare_calibration_input(model, dataloader, device)
    
    if attention_mask is not None: attention_mask = attention_mask.to(device)
    if position_ids is not None: position_ids = position_ids.to(device)
    
    # Move rotary_emb to GPU for the whole pruning process
    model.model.rotary_emb.to(device)

    layers = model.model.layers
    for i in range(len(layers)):
        print(f"Pruning Layer {i}...")
        layer = layers[i].to(device)
        subset = find_layers(layer)

        handles = []
        for name in subset:
            def get_hook(name):
                def hook(_, inp, out):
                    if not hasattr(subset[name], 'scaler_row'):
                        subset[name].scaler_row = torch.zeros(subset[name].weight.shape[1], device=device)
                    subset[name].scaler_row += torch.norm(inp[0].data, p=2, dim=[0, 1]).float()**2
                return hook
            handles.append(subset[name].register_forward_hook(get_hook(name)))
        
        for j in range(args.nsamples):
            with torch.no_grad():
                curr_in = inps[j].unsqueeze(0).to(device)
                cos, sin = model.model.rotary_emb(curr_in, position_ids)
                layer(curr_in, attention_mask=attention_mask, position_ids=position_ids, position_embeddings=(cos, sin))
            
        for h in handles: h.remove()

        for name in subset:
            W = subset[name].weight.data
            scaling_factor = torch.sqrt(subset[name].scaler_row).to(W.dtype).reshape((1, -1))
            W_metric = torch.abs(W) * scaling_factor
            thresh = torch.sort(W_metric.flatten())[0][int(W_metric.numel() * args.sparsity_ratio)]
            W[W_metric <= thresh] = 0
            del subset[name].scaler_row
            torch.cuda.empty_cache()

        for j in range(args.nsamples):
            with torch.no_grad():
                curr_in = inps[j].unsqueeze(0).to(device)
                cos, sin = model.model.rotary_emb(curr_in, position_ids)
                layer_out = layer(curr_in, attention_mask=attention_mask, position_ids=position_ids, position_embeddings=(cos, sin))[0]
                outs[j] = layer_out.detach().cpu()
        
        inps.data = outs.data 
        layers[i] = layer.cpu()
        del layer
        gc.collect()
        torch.cuda.empty_cache()

    print("Pruning finished.")

def check_sparsity(model):
    all_weights = 0
    zero_weights = 0
    for name, p in model.named_parameters():
        if 'weight' in name:
            all_weights += p.numel()
            zero_weights += (p == 0).sum().item()
    return float(zero_weights) / all_weights if all_weights > 0 else 0.0

Overwriting lib/prune.py


In [7]:
import os

file_path = '/kaggle/working/wanda/lib/prune.py'

if os.path.exists(file_path):
    with open(file_path, 'r') as f:
        content = f.read()

    # This version uses a safer attribute lookup to prevent RecursionError
    safe_catcher = """class Catcher(nn.Module):
        def __init__(self, module):
            super().__init__()
            self.__dict__['module'] = module
        def __getattr__(self, name):
            return getattr(self.module, name)
        def forward(self, inp, **kwargs):"""

    # We use a regex to replace the entire old Catcher block with the safe one
    import re
    pattern = r"class Catcher\(nn\.Module\):.*?def forward\(self, inp, \*\*kwargs\):"
    
    # Clean up the content to ensure we are replacing the right block
    fixed_content = re.sub(pattern, safe_catcher, content, flags=re.DOTALL)

    with open(file_path, 'w') as f:
        f.write(fixed_content)
    print("✅ Fixed RecursionError in Catcher. The module is now safely impersonated.")
else:
    print(f"❌ Error: File not found at {file_path}")

✅ Fixed RecursionError in Catcher. The module is now safely impersonated.


In [8]:
!touch lib/__init__.py

## Wanda 25

In [29]:
import torch
import gc
gc.collect()
torch.cuda.empty_cache()
# This helps prevent the fragmentation mentioned in your error log
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True


import time

start = time.time()

!python main.py \
    --model "Qwen/Qwen3-4B" \
    --nsamples 64 \
    --sparsity_ratio 0.25 \
    --seed {SEED} \
    --save_model Qwen3-4B-WANDA-25

end = time.time()

elapsed = end - start

hours = int(elapsed // 3600)
minutes = int((elapsed % 3600) // 60)
seconds = elapsed % 60
milliseconds = elapsed * 1000

print(f"Time taken: {hours}h {minutes}m {seconds:.2f}s")
print(f"Milliseconds: {milliseconds:.2f} ms")


env: PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
Loading weights: 100%|████████████████████████| 398/398 [00:03<00:00, 99.64it/s]
Pruning starts (Float16 mode)...
loading calibration data
Token indices sequence length is longer than the specified maximum sequence length for this model (526359 > 131072). Running this sequence through the model will result in indexing errors
Pruning Layer 0...
Pruning Layer 1...
Pruning Layer 2...
Pruning Layer 3...
Pruning Layer 4...
Pruning Layer 5...
Pruning Layer 6...
Pruning Layer 7...
Pruning Layer 8...
Pruning Layer 9...
Pruning Layer 10...
Pruning Layer 11...
Pruning Layer 12...
Pruning Layer 13...
Pruning Layer 14...
Pruning Layer 15...
Pruning Layer 16...
Pruning Layer 17...
Pruning Layer 18...
Pruning Layer 19...
Pruning Layer 20...
Pruning Layer 21...
Pruning Layer 22...
Pruning Layer 23...
Pruning Layer 24...
Pruning Layer 25...
Pruning Layer 26...
Pruning Layer 27...
Pruning Layer 28...
Pruning Layer 29...
Pruning Layer 30...
Pruning L

In [33]:
import torch
from transformers import AutoModelForCausalLM

def check_pruned_model(model_path):
    model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype="auto", device_map="cpu")
    
    total_params = sum(p.numel() for p in model.parameters())
    # count_nonzero is the key for unstructured pruning
    active_params = sum(torch.count_nonzero(p).item() for p in model.parameters())
    
    sparsity = 100 * (1 - active_params / total_params)
    print(f"Model: {model_path}")
    print(f"Total Params: {total_params / 1e9:.2f}B")
    print(f"Active Params: {active_params / 1e9:.2f}B")
    print(f"Effective Sparsity: {sparsity:.2f}%")

# Usage
check_pruned_model(OUTPUT_DIR_25)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Model: /kaggle/working/wanda/wanda/Qwen3-4B-WANDA-25
Total Params: 4.02B
Active Params: 3.11B
Effective Sparsity: 22.59%


In [34]:
import torch
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(OUTPUT_DIR_25, torch_dtype=torch.float16, device_map="cpu")

# Check a specific layer (e.g., Layer 5 Down Proj)
weight = model.model.layers[5].mlp.down_proj.weight.data
zeros = (weight == 0).sum().item()
total = weight.numel()

print(f"Layer 5 Down Proj Sparsity: {zeros/total*100:.2f}%")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Layer 5 Down Proj Sparsity: 25.00%


In [36]:
# create repo in organization
create_repo(REPO_ID_25, repo_type="model", exist_ok=True)

# upload to organization repo
upload_folder(
    repo_id=REPO_ID_25,
    repo_type="model",
    folder_path=OUTPUT_DIR_25,
    commit_message="Upload WANAD Pruningmodel"
)

print("Upload complete to organization!")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Upload complete to organization!


In [40]:
!rm -rf /kaggle/working/wanda/Qwen3-4B-WANDA-25

## Wanda 50

In [13]:
import torch
import gc
gc.collect()
torch.cuda.empty_cache()
# This helps prevent the fragmentation mentioned in your error log
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True


import time

start = time.time()

!python main.py \
    --model "Qwen/Qwen3-4B" \
    --nsamples 64 \
    --sparsity_ratio 0.5 \
    --seed {SEED} \
    --save_model Qwen3-4B-WANDA-50

end = time.time()

elapsed = end - start

hours = int(elapsed // 3600)
minutes = int((elapsed % 3600) // 60)
seconds = elapsed % 60
milliseconds = elapsed * 1000

print(f"Time taken: {hours}h {minutes}m {seconds:.2f}s")
print(f"Milliseconds: {milliseconds:.2f} ms")


env: PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
Loading weights: 100%|███████████████████████| 398/398 [00:03<00:00, 109.50it/s]
Pruning starts (Float16 mode)...
loading calibration data
Loading C4 training shard...
Token indices sequence length is longer than the specified maximum sequence length for this model (526359 > 131072). Running this sequence through the model will result in indexing errors
Pruning Layer 0...
Pruning Layer 1...
Pruning Layer 2...
Pruning Layer 3...
Pruning Layer 4...
Pruning Layer 5...
Pruning Layer 6...
Pruning Layer 7...
Pruning Layer 8...
Pruning Layer 9...
Pruning Layer 10...
Pruning Layer 11...
Pruning Layer 12...
Pruning Layer 13...
Pruning Layer 14...
Pruning Layer 15...
Pruning Layer 16...
Pruning Layer 17...
Pruning Layer 18...
Pruning Layer 19...
Pruning Layer 20...
Pruning Layer 21...
Pruning Layer 22...
Pruning Layer 23...
Pruning Layer 24...
Pruning Layer 25...
Pruning Layer 26...
Pruning Layer 27...
Pruning Layer 28...
Pruning Layer 29...


In [15]:
import torch
from transformers import AutoModelForCausalLM

def check_pruned_model(model_path):
    model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype="auto", device_map="cpu")
    
    total_params = sum(p.numel() for p in model.parameters())
    # count_nonzero is the key for unstructured pruning
    active_params = sum(torch.count_nonzero(p).item() for p in model.parameters())
    
    sparsity = 100 * (1 - active_params / total_params)
    print(f"Model: {model_path}")
    print(f"Total Params: {total_params / 1e9:.2f}B")
    print(f"Active Params: {active_params / 1e9:.2f}B")
    print(f"Effective Sparsity: {sparsity:.2f}%")

# Usage
check_pruned_model(OUTPUT_DIR_50)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Model: /kaggle/working/wanda/Qwen3-4B-WANDA-25
Total Params: 4.02B
Active Params: 2.21B
Effective Sparsity: 45.17%


In [16]:
import torch
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(OUTPUT_DIR_50, torch_dtype=torch.float16, device_map="cpu")

# Check a specific layer (e.g., Layer 5 Down Proj)
weight = model.model.layers[5].mlp.down_proj.weight.data
zeros = (weight == 0).sum().item()
total = weight.numel()

print(f"Layer 5 Down Proj Sparsity: {zeros/total*100:.2f}%")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Layer 5 Down Proj Sparsity: 50.02%


In [17]:
# create repo in organization
create_repo(REPO_ID_50, repo_type="model", exist_ok=True)

# upload to organization repo
upload_folder(
    repo_id=REPO_ID_50,
    repo_type="model",
    folder_path=OUTPUT_DIR_50,
    commit_message="Upload WANAD Pruningmodel"
)

print("Upload complete to organization!")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Upload complete to organization!


In [18]:
!rm -rf /kaggle/working/wanda/Qwen3-4B-WANDA-50

## Wanda 70

In [9]:
import torch
import gc
gc.collect()
torch.cuda.empty_cache()
# This helps prevent the fragmentation mentioned in your error log
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True


import time

start = time.time()

!python main.py \
    --model "Qwen/Qwen3-4B" \
    --nsamples 64 \
    --sparsity_ratio 0.7 \
    --seed {SEED} \
    --save_model Qwen3-4B-WANDA-70

end = time.time()

elapsed = end - start

hours = int(elapsed // 3600)
minutes = int((elapsed % 3600) // 60)
seconds = elapsed % 60
milliseconds = elapsed * 1000

print(f"Time taken: {hours}h {minutes}m {seconds:.2f}s")
print(f"Milliseconds: {milliseconds:.2f} ms")


env: PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
config.json: 100%|█████████████████████████████| 726/726 [00:00<00:00, 2.56MB/s]
model.safetensors.index.json: 32.8kB [00:00, 23.1MB/s]
Fetching 3 files: 100%|███████████████████████████| 3/3 [00:23<00:00,  7.87s/it]
Download complete: 100%|████████████████████| 8.04G/8.04G [00:23<00:00, 173MB/s]
Loading weights: 100%|███████████████████████| 398/398 [00:03<00:00, 107.47it/s]

generation_config.json: 100%|███████████████████| 239/239 [00:00<00:00, 873kB/s]

config.json: 100%|█████████████████████████████| 726/726 [00:00<00:00, 3.32MB/s]

tokenizer_config.json: 9.73kB [00:00, 8.90MB/s]

vocab.json: 2.78MB [00:00, 87.0MB/s]

merges.txt: 1.67MB [00:00, 70.1MB/s]

tokenizer.json:   0%|                               | 0.00/11.4M [00:00<?, ?B/s]
tokenizer.json: 100%|██████████████████████| 11.4M/11.4M [00:00<00:00, 56.4MB/s]
Download complete: 100%|████████████████████| 8.04G/8.04G [00:30<00:00, 263MB/s]
Pruning starts (Float16 mode)...
l

In [10]:
import torch
from transformers import AutoModelForCausalLM

def check_pruned_model(model_path):
    model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype="auto", device_map="cpu")
    
    total_params = sum(p.numel() for p in model.parameters())
    # count_nonzero is the key for unstructured pruning
    active_params = sum(torch.count_nonzero(p).item() for p in model.parameters())
    
    sparsity = 100 * (1 - active_params / total_params)
    print(f"Model: {model_path}")
    print(f"Total Params: {total_params / 1e9:.2f}B")
    print(f"Active Params: {active_params / 1e9:.2f}B")
    print(f"Effective Sparsity: {sparsity:.2f}%")

# Usage
check_pruned_model(OUTPUT_DIR_70)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Model: /kaggle/working/wanda/Qwen3-4B-WANDA-70
Total Params: 4.02B
Active Params: 1.48B
Effective Sparsity: 63.24%


In [11]:
import torch
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(OUTPUT_DIR_70, torch_dtype=torch.float16, device_map="cpu")

# Check a specific layer (e.g., Layer 5 Down Proj)
weight = model.model.layers[5].mlp.down_proj.weight.data
zeros = (weight == 0).sum().item()
total = weight.numel()

print(f"Layer 5 Down Proj Sparsity: {zeros/total*100:.2f}%")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Layer 5 Down Proj Sparsity: 70.01%


In [12]:
# create repo in organization
create_repo(REPO_ID_70, repo_type="model", exist_ok=True)

# upload to organization repo
upload_folder(
    repo_id=REPO_ID_70,
    repo_type="model",
    folder_path=OUTPUT_DIR_70,
    commit_message="Upload WANAD Pruningmodel"
)

print("Upload complete to organization!")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Upload complete to organization!


In [ ]:
!rm -rf /kaggle/working/wanda/Qwen3-4B-WANDA-70

## Test the model

In [ ]:
import torch
from transformers import AutoTokenizer

# 1. Setup
model_path = "/kaggle/working/wanda/Qwen3-4B-WANDA-70"
tokenizer = AutoTokenizer.from_pretrained(model_path)
prompt = "The best way to prune a neural network is"

# 2. Ensure model is on GPU
model.to("cuda")

# 3. Manually tokenize and move inputs to CUDA
# This ensures the 'index' (input_ids) is on the same device as the weights
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# 4. Generate using the model directly (more reliable than pipeline here)
with torch.no_grad():
    output_ids = model.generate(
        **inputs, 
        max_new_tokens=50, 
        do_sample=True, 
        temperature=0.7,
        pad_token_id=tokenizer.eos_token_id
    )

# 5. Decode and Print
response = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print(response)

In [ ]:
# Step 1: The Perplexity Script

import torch
from tqdm import tqdm
from datasets import load_dataset

def evaluate_perplexity(model, tokenizer, dataset_name="wikitext", dataset_config="wikitext-2-raw-v1"):
    # 1. Load the test set
    test_data = load_dataset(dataset_name, dataset_config, split="test")
    encodings = tokenizer("\n\n".join(test_data["text"]), return_tensors="pt")

    max_length = 2048 # Llama 3.2 context window
    stride = 512
    seq_len = encodings.input_ids.size(1)

    nlls = []
    prev_end_loc = 0
    
    print(f"Calculating Perplexity for {seq_len} tokens...")
    
    for begin_loc in tqdm(range(0, seq_len, stride)):
        end_loc = min(begin_loc + max_length, seq_len)
        trg_len = end_loc - prev_end_loc  # may be different from stride on last loop
        input_ids = encodings.input_ids[:, begin_loc:end_loc].to("cuda")
        target_ids = input_ids.clone()
        target_ids[:, :-trg_len] = -100

        with torch.no_grad():
            outputs = model(input_ids, labels=target_ids)
            # loss is calculated using CrossEntropyLoss which averages over valid labels
            neg_log_likelihood = outputs.loss

        nlls.append(neg_log_likelihood * trg_len)

        prev_end_loc = end_loc
        if end_loc == seq_len:
            break

    ppl = torch.exp(torch.stack(nlls).sum() / end_loc)
    return ppl.item()

# Run the evaluation
# ppl_value = evaluate_perplexity(model, tokenizer)
# print(f"\nFinal Perplexity: {ppl_value:.4f}")

In [ ]:
import time

def mobile_sim_benchmark(model, tokenizer, prompt="Explain AI in 10 words."):
    # 1. Force CPU and limit threads to simulate mobile processor
    torch.set_num_threads(4) 
    model.to("cpu")
    
    inputs = tokenizer(prompt, return_tensors="pt")
    
    # Warm start
    _ = model.generate(**inputs, max_new_tokens=1)
    
    # Start Timing
    start = time.time()
    outputs = model.generate(**inputs, max_new_tokens=30, do_sample=True)
    end = time.time()
    
    # Calculate Metrics
    tokens_gen = len(outputs[0]) - len(inputs[0])
    tps = tokens_gen / (end - start)
    
    print(f"--- Mobile Simulation Result ---")
    print(f"Tokens/Sec (TPS): {tps:.2f}")
    print(f"Latency: {end - start:.2f}s")
    
    if tps < 5:
        print("Status: Too slow for phone (Needs 4-bit Quantization)")
    else:
        print("Status: Good for Edge!")

# mobile_sim_benchmark(model, tokenizer)

In [ ]:
# !python main.py \
#     --model "meta-llama/Llama-3.2-3B-Instruct" \
#     --prune_method wanda \
#     --sparsity_ratio 0.25 \
#     --sparsity_type unstructured \
#     --save out/llama_3.2_pruned/

# !python main.py \
#     --model meta-llama/Llama-3.2-3B-Instruct \
#     --prune_method wanda \
#     --sparsity_ratio 0.5 \
#     --sparsity_type unstructured \
#     --save_model out/llama_3.2_pruned/

# Important Kaggle Tip: Since you have cuda:0 detected now, the process will be much faster. 
# However, Llama 3.2 3B uses a lot of memory during the "Wanda" phase (where it calculates the $X^TX$ Hessian matrix). 
# If you hit an Out of Memory (OOM) error, reduce the --nsamples from the default 128 to 64.

# !python main.py \
#     --model meta-llama/Llama-3.2-3B-Instruct \
#     --prune_method wanda \
#     --sparsity_ratio 0.25 \
#     --sparsity_type unstructured \
#     --nsamples 32 \
#     --save out/llama_3_2_pruned/